In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv()


os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
# import dependencies
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

In [16]:
#pip install llama-index
#pip install python-docx
#pip install chromadb

In [11]:
from docx import Document as WordDocument
from llama_index.core import (VectorStoreIndex,Document,Settings)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = None

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Read Word document
doc = WordDocument("employee_benefits.docx")

text = "\n".join(para.text for para in doc.paragraphs if para.text.strip())

# Create LlamaIndex document
documents = [Document(text=text,metadata={"source": "employee_benefits.docx"})]

# Build index
index = VectorStoreIndex.from_documents(documents)

# Save index
index.storage_context.persist(persist_dir="./storage")

print("Document indexed successfully.")

LLM is explicitly disabled. Using MockLLM.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Document indexed successfully.


In [ ]:
#llm1 = ChatGroq(model="openai/gpt-oss-120b",temperature=0,max_tokens=500, groq_api_key=os.getenv("GROQ_API_KEY"))
#llm1.invoke("what is 2+2")

AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks a simple math question: "what is 2+2". Answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 77, 'total_tokens': 118, 'completion_time': 0.085964119, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.003777117, 'prompt_tokens_details': None, 'queue_time': 0.340820775, 'total_time': 0.089741236}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1433b-8d2c-7cc3-bb53-f286f859dde2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 41, 'total_tokens': 118, 'output_token_details': {'reasoning': 22}})

In [ ]:
from llama_index.core import (StorageContext,load_index_from_storage)

# Load index
storage_context = StorageContext.from_defaults(persist_dir="./storage")

i-123456
ndex = load_index_from_storage(storage_context)

retriever = index.as_retriever(similarity_top_k=3)

# Groq LLM
llm = ChatGroq(model="qwen/qwen3.8-27b",temperature=0,max_tokens=500)

prompt_template = PromptTemplate(input_variables=["context", "question"],
    template="""
You are a helpful document assistant.
Answer only from the supplied context.
If the answer is not present in the document,
respond with:
"I could not find this information in the document."
Context:
{context}
Question:
{question}
Answer:
"""
)
def ask_question(question):
    nodes = retriever.retrieve(question)
    context = "\n\n".join(node.text for node in nodes)

    final_prompt = prompt_template.format(
        context=context,
        question=question
    )

    response = llm.invoke(final_prompt)

    print("\nAnswer:")
    print(response.content)

    print("\nSources:")
    for node in nodes:
        print(node.metadata.get(
                "source",
                "document.docx"
            )
        )

while True:

    user_query = input("\nAsk a question (type exit to quit): ")

    if user_query.lower() == "exit":
        break

    ask_question(user_query)


Answer:
Sick Leaves: 80 days per year

Sources:
employee_benefits.docx
employee_benefits.docx

Answer:
Sick Leaves: 80 days per year

Sources:
employee_benefits.docx
employee_benefits.docx

Answer:
Leaves Encashment: 100 days upto a maximum 200 days leaves

Sources:
employee_benefits.docx
employee_benefits.docx
